In [1]:
!pip install albumentations opencv-python tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   -- ------------------------------------- 2.1/40.2 MB 9.8 MB/s eta 0:00:04
   ---- ----------------------------------- 4.5/40.2 MB 9.9 MB/s eta 0:00:04
   ----- ---------------------------------- 5.8/40.2 MB 10.1 MB/s eta 0:00:04
   ------- -------------------------------- 7.6/40.2 MB 8.9 MB/s eta 0:00:04
   -------- ------------------------------- 8.9/40.2 MB 8.3 MB/s eta 0:00:04
   ---------- ----------------------------- 10.5/40.2 MB 7.9 MB/s eta 0:00:04
   ----------- ---------------------------- 11.8/40.2 MB 7.8 MB/s eta 0:00:04
   ------------- -------------------------- 13.1/40.2 MB 7.5 MB/s eta 0:00:04
   -------------- ------------------------- 14.7/40.2 MB 7.5 MB/s eta 0:00:04
   --------------- ------------------------ 16.0/40.2 MB 7.3 MB/s eta 0:00:04
   ----------------- ---------------------- 17.3/40.2 MB 7.3 MB/s eta 0:00:04
   ------------

In [2]:
# Importar librerías

from pathlib import Path
import shutil
import cv2
import albumentations as A
from tqdm import tqdm

In [21]:
# Configuración de rutas y clases

INPUT_DIR = Path("dataset_split")
OUTPUT_DIR = Path("dataset_augmented")

CLASSES = ["multirotor", "ala_fija", "helicoptero", "no_uav"]

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

N_AUGMENTED_PER_IMAGE = 2

In [22]:
# Definir las transformaciones

augmentation = A.Compose([
    A.HorizontalFlip(p=0.5),

    A.Affine(
        scale=(0.90, 1.10),              # zoom suave: entre -10 % y +10 %
        translate_percent=(-0.05, 0.05), # desplazamiento suave
        rotate=(-10, 10),                # rotación entre -10º y +10º
        shear=0,                         # sin deformación inclinada
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.7
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.5
    ),

    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=10,
        val_shift_limit=10,
        p=0.3
    )
])

In [23]:
# Funciones auxiliares

def is_image(file_path):
    return file_path.suffix.lower() in IMAGE_EXTENSIONS


def prepare_output_folder():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def copy_val_and_test():
    for split in ["val", "test"]:
        for class_name in CLASSES:
            input_class_dir = INPUT_DIR / split / class_name
            output_class_dir = OUTPUT_DIR / split / class_name
            output_class_dir.mkdir(parents=True, exist_ok=True)

            for img_path in input_class_dir.iterdir():
                if is_image(img_path):
                    shutil.copy2(img_path, output_class_dir / img_path.name)


def augment_train():
    for class_name in CLASSES:
        input_class_dir = INPUT_DIR / "train" / class_name
        output_class_dir = OUTPUT_DIR / "train" / class_name
        output_class_dir.mkdir(parents=True, exist_ok=True)

        image_paths = [p for p in input_class_dir.iterdir() if is_image(p)]

        for img_path in tqdm(image_paths, desc=f"Aumentando {class_name}"):
            image = cv2.imread(str(img_path))

            if image is None:
                print(f"No se pudo leer: {img_path}")
                continue

            # Guardar imagen original
            shutil.copy2(img_path, output_class_dir / img_path.name)

            # Convertir BGR a RGB para Albumentations
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Generar imágenes aumentadas
            for i in range(N_AUGMENTED_PER_IMAGE):
                augmented = augmentation(image=image_rgb)
                augmented_image_rgb = augmented["image"]

                # Volver a BGR para guardar con OpenCV
                augmented_image_bgr = cv2.cvtColor(
                    augmented_image_rgb,
                    cv2.COLOR_RGB2BGR
                )

                new_filename = f"{img_path.stem}_aug_{i+1}{img_path.suffix}"
                output_path = output_class_dir / new_filename

                cv2.imwrite(str(output_path), augmented_image_bgr)

In [24]:
# Ejecución del aumento de datos

prepare_output_folder()
augment_train()
copy_val_and_test()

print("Dataset aumentado creado correctamente en:", OUTPUT_DIR)

Aumentando no_uav: 100%|██████████| 175/175 [00:17<00:00,  9.90it/s]


Dataset aumentado creado correctamente en: dataset_augmented
